# FiNTA — Model Evaluation

Đánh giá model sau khi training. Chạy 1 lần duy nhất trên test set.

In [ ]:
import json
import torch
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_PATH = "./checkpoints/sentiment/best_model"  # hoặc HF repo ID
TEST_PATH = "../data/splits/test.json"

LABEL_NAMES = ["Bullish", "Bearish", "Uncertainty", "Hype/FOMO", "Panic", "Contrarian"]
LABEL2ID = {label: i for i, label in enumerate(LABEL_NAMES)}

In [ ]:
# Load model và test set
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

with open(TEST_PATH, encoding="utf-8") as f:
    test_rows = json.load(f)

print(f"Test set: {len(test_rows)} bài")

In [ ]:
# Inference
all_preds, all_labels = [], []

for row in test_rows:
    text = f"Tiêu đề: {row.get('title', '')}\n\n{row.get('raw_content', '')}"
    enc = tokenizer(text, max_length=256, padding="max_length",
                    truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    pred = logits.argmax().item()
    all_preds.append(pred)
    all_labels.append(LABEL2ID[row["sentiment_label"]])

print("Inference hoàn tất!")

In [ ]:
# Kết quả
print("CLASSIFICATION REPORT (Test Set):")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES, digits=4))

from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print("\nSLA Check:")
print(f"  Accuracy: {acc:.4f} {'PASS' if acc >= 0.80 else 'FAIL'} (target: 0.80)")
print(f"  Macro F1: {macro_f1:.4f} {'PASS' if macro_f1 >= 0.78 else 'FAIL'} (target: 0.78)")